In [3]:
import pandas as pd
import numpy as np
from sklearn.feature_extraction.text import CountVectorizer
import re
import nltk
from nltk.tokenize import word_tokenize
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer
import string
from collections import Counter
from textblob import TextBlob, Word, Blobber
from textblob.classifiers import NaiveBayesClassifier
from textblob.taggers import NLTKTagger

In [4]:
# Download required NLTK resources
nltk.download('punkt')
nltk.download('punkt_tab')
nltk.download('stopwords')
nltk.download('wordnet')

[nltk_data] Downloading package punkt to /home/twjcompute/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt.zip.
[nltk_data] Downloading package punkt_tab to
[nltk_data]     /home/twjcompute/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!
[nltk_data] Downloading package stopwords to
[nltk_data]     /home/twjcompute/nltk_data...
[nltk_data]   Unzipping corpora/stopwords.zip.
[nltk_data] Downloading package wordnet to
[nltk_data]     /home/twjcompute/nltk_data...


True

In [ ]:
def create_corpus(input_file):
    """
    Step 1: Create corpus from input file
    """
    # Read the file and split by lines
    with open(input_file, 'r', encoding='utf-8') as f:
        lines = f.readlines()

In [ ]:
 # Initialize lists to store data
    user_ids = []
    tweets = []

In [ ]:
 # Extract user IDs and tweets
    for line in lines:
        # Extract user ID (assuming format "node/tweet")
        match = re.match(r'(\w+)\)(.*)', line.strip())
        if match:
            user_id, tweet = match.groups()
            user_ids.append(user_id)
            tweets.append(tweet)
    

In [ ]:
# Create DataFrame
    df = pd.DataFrame({
        'user_id': user_ids,
        'original_tweet': tweets
    })
    
    return df

In [12]:
 """
    Step 2: Clean the corpus
    """
    def clean_text(text):
        # Convert to lowercase
        text = text.lower()
        
        # Remove URLs
        text = re.sub(r'http\S+|www\S+|https\S+', '', text, flags=re.MULTILINE)
        
        # Remove user mentions (@username)
        text = re.sub(r'@\w+', '', text)
        
        # Remove hashtags symbols (keep the text)
        text = re.sub(r'#', '', text)
        text = re.sub(r'’', '', text)
        
        
        # Remove RT (retweet) symbol
        text = re.sub(r'^RT[\s]+', '', text)
        
        # Remove punctuation
        text = text.translate(str.maketrans('', '', string.punctuation))
        
        # Remove numbers
        text = re.sub(r'\d+', '', text)
        
        # Remove extra whitespace
        text = ' '.join(text.split())
         # Tokenization
        tokens = word_tokenize(text)
        
        # Remove stopwords
        stop_words = set(stopwords.words('english'))
        stop_words.add('amp')  
        stop_words.add('node')
        tokens = [token for token in tokens if token not in stop_words]
        
        # Lemmatization
        lemmatizer = WordNetLemmatizer()
        tokens = [lemmatizer.lemmatize(token) for token in tokens]
        
         # Join tokens back into text
        cleaned_text = ' '.join(tokens)
        return cleaned_text
    
    # Apply cleaning to all tweets
    df['cleaned_tweet'] = df['original_tweet'].apply(clean_text)

In [ ]:
#Step 3: Create Document-Feature Matrix (DFM)

    def create_dfm(df, min_df=2, max_features=300):
        # Initialize the CountVectorizer
    vectorizer = CountVectorizer(min_df=min_df, 
                               max_features=max_features,
                               token_pattern=r'\b\w+\b')  # Matches whole words only
     # Create the DFM
    dfm = vectorizer.fit_transform(df['cleaned_tweet'])

In [ ]:
 # Get feature names (vocabulary)
    feature_names = vectorizer.get_feature_names_out()

In [ ]:
 # Convert to DataFrame for easier handling
    dfm_df = pd.DataFrame(dfm.toarray(),
                         index=df['user_id'],
                         columns=feature_names)
    return dfm_df, feature_names


In [ ]:
# ACTUALLY RUNNING THE PIPELINE 

input_file = "community_5_1K_central_nodes.txt"

 #Step 1: Create corpus
print("Step 1: Creating corpus...")
df = create_corpus(input_file)

#print first 5 rows of the dataframe
print(df.head())


In [ ]:
# Step 2: Clean corpus
print("Step 2: Cleaning corpus...")
cleaned_df = clean_corpus(df)

# Print sample results
print("\nSample of cleaned tweets:")
print(cleaned_df[['user_id', 'cleaned_tweet']].head())

In [ ]:
#Step 3 Stats from the Corpus
def analyze_tokens(df, column='cleaned_tweet'):
    # Get all tokens from all documents
    all_tokens = []
    token_counts_per_doc = []
    
    for text in df[column]:
        tokens = word_tokenize(text)
        all_tokens.extend(tokens)
        token_counts_per_doc.append(len(tokens))
    
    # Create frequency distribution
    token_freq = Counter(all_tokens)
    
    # Calculate statistics
    stats = {
        'total_tokens': len(all_tokens),
        'unique_tokens': len(set(all_tokens)),
        'average_tokens_per_doc': np.mean(token_counts_per_doc),
        'median_tokens_per_doc': np.median(token_counts_per_doc),
        'min_tokens': min(token_counts_per_doc),
        'max_tokens': max(token_counts_per_doc)
    }
    
    return stats, token_freq

def print_token_analysis(stats, token_freq, top_n=10):

In [ ]:
 #Print token analysis results
print("\nToken Statistics:")
    print(f"Total tokens: {stats['total_tokens']:,}")
    print(f"Unique tokens: {stats['unique_tokens']:,}")
    print(f"Average tokens per document: {stats['average_tokens_per_doc']:.2f}")
    print(f"Median tokens per document: {stats['median_tokens_per_doc']:.2f}")
    print(f"Min tokens in a document: {stats['min_tokens']}")
    print(f"Max tokens in a document: {stats['max_tokens']}")
    
    print(f"\nTop {top_n} most frequent tokens:")
    for token, count in token_freq.most_common(top_n):
        print(f"{token}: {count}")


In [ ]:
# Analyze tokens before DFM creation
print("\n=== Token Analysis Before Cleaning ===")
original_stats, original_freq = analyze_tokens(df, 'original_tweet')
print_token_analysis(original_stats, original_freq)

print("\n=== Token Analysis After Cleaning ===")
cleaned_stats, cleaned_freq = analyze_tokens(cleaned_df, 'cleaned_tweet')
print_token_analysis(cleaned_stats, cleaned_freq)
exit()

In [ ]:
# Step 4: Create DFM
print("Step 3: Creating Document-Feature Matrix...")
dfm, features = create_dfm(cleaned_df, min_df=min_df, max_features=max_features)


In [ ]:
print("\nShape of DFM:", dfm.shape)
print("\nSample of DFM (first 5 features):")
print(dfm.iloc[:5, :5])